# 03 — Exploratory Data Analysis (Phase 2)

Scope:
- Descriptive statistics and visualisations on **train + validation only**.
- Outliers, associations, and patterns.
- **Linear DR (PCA)** and **non-linear DR (UMAP)**.
- Initial insights (model-development EDA is kept separate from the held-out test set).

Inputs:
- `data/processed/accepted_regression.parquet`
- `data/processed/accepted_classification.parquet`
- `outputs/split_ids.json` (chronological cutoffs)

Outputs: figures in `outputs/figures/` and tables in `outputs/tables/`.

In [1]:
import sys, json, logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.eda_utils import (
    reproducible_sample, describe_numeric, plot_histograms, plot_scatter,
    plot_correlation_heatmap, run_pca, run_umap,
)
from src.splits import chronological_split

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
sns.set_theme(style='whitegrid')
OUT_FIG = PROJECT_ROOT / 'outputs' / 'figures'; OUT_FIG.mkdir(parents=True, exist_ok=True)
OUT_TABLES = PROJECT_ROOT / 'outputs' / 'tables'; OUT_TABLES.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT

PosixPath('/Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados')

## 1. Load cleaned data and reuse the frozen split

No re-splitting happens here. We only reuse the cutoffs recorded in Notebook 02.

In [2]:
with open(PROJECT_ROOT / 'outputs' / 'split_definition.json') as f:
    split_cfg = json.load(f)
split_cfg

{'date_col': 'issue_d', 'train_end': '2014-12', 'val_end': '2016-12'}

In [3]:
acc_reg = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'accepted_regression.parquet')
acc_clf = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'accepted_classification.parquet')
print('acc_reg:', acc_reg.shape, '| acc_clf:', acc_clf.shape)

acc_reg: (2260668, 152) | acc_clf: (1379602, 152)


In [4]:
splits_reg = chronological_split(acc_reg, 'issue_d', split_cfg['train_end'], split_cfg['val_end'])
splits_clf = chronological_split(acc_clf, 'issue_d', split_cfg['train_end'], split_cfg['val_end'])

# EDA is done on train + validation ONLY. Test is untouched.
eda_reg = pd.concat([splits_reg.train, splits_reg.val], ignore_index=True)
eda_clf = pd.concat([splits_clf.train, splits_clf.val], ignore_index=True)
print('EDA regression rows:', len(eda_reg))
print('EDA classification rows:', len(eda_clf))

EDA regression rows: 1321847
EDA classification rows: 1129813


## 2. Define the EDA column sets

Only variables that are legitimately eligible at the prediction time are analysed
here. Leakage columns, `grade`, `sub_grade` and `installment` are excluded from
the regression EDA because they are banned by the specification.

In [5]:
EDA_NUMERIC = [
    'loan_amnt', 'term', 'annual_inc', 'dti', 'fico_range_low', 'fico_range_high',
    'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'total_acc', 'pub_rec',
    'pub_rec_bankruptcies', 'mort_acc', 'revol_bal', 'revol_util',
    'tot_cur_bal', 'tot_hi_cred_lim', 'bc_util', 'bc_open_to_buy',
    'mo_sin_old_rev_tl_op', 'num_actv_rev_tl', 'pct_tl_nvr_dlq',
]
EDA_NUMERIC = [c for c in EDA_NUMERIC if c in eda_reg.columns]
if 'int_rate' in eda_reg.columns:
    EDA_NUMERIC_REG = EDA_NUMERIC + ['int_rate']
else:
    EDA_NUMERIC_REG = EDA_NUMERIC
EDA_NUMERIC_REG

['loan_amnt',
 'term',
 'annual_inc',
 'dti',
 'fico_range_low',
 'fico_range_high',
 'delinq_2yrs',
 'inq_last_6mths',
 'open_acc',
 'total_acc',
 'pub_rec',
 'pub_rec_bankruptcies',
 'mort_acc',
 'revol_bal',
 'revol_util',
 'tot_cur_bal',
 'tot_hi_cred_lim',
 'bc_util',
 'bc_open_to_buy',
 'mo_sin_old_rev_tl_op',
 'num_actv_rev_tl',
 'pct_tl_nvr_dlq',
 'int_rate']

## 3. Descriptive statistics

Extended describe (count, missing, mean, std, quartiles, skew, kurtosis).
Saved to CSV for the mid-journey report.

In [6]:
desc_reg = describe_numeric(eda_reg, EDA_NUMERIC_REG)
desc_clf = describe_numeric(eda_clf, EDA_NUMERIC)
desc_reg.to_csv(OUT_TABLES / 'eda_describe_regression.csv')
desc_clf.to_csv(OUT_TABLES / 'eda_describe_classification.csv')
desc_reg.head(15)

,count,mean,std,min,25%,50%,75%,max,missing_pct,skew,kurtosis
loan_amnt,1321847.0,14748.2240,8622.1427,500.0,8000.00,12900.00,20000.00,40000.0,0.0000,0.7092,-0.2190
term,1321847.0,42.8483,10.8379,36.0,36.00,36.00,60.00,60.0,0.0000,0.9507,-1.0962
annual_inc,1321843.0,76495.9420,69111.9196,0.0,46000.00,65000.00,90000.00,9573072.0,0.0003,44.1991,4426.1994
dti,1321782.0,18.3746,9.1209,-1.0,12.04,17.83,24.24,999.0,0.0049,11.2446,1034.5501
fico_range_low,1321847.0,695.0868,30.7992,610.0,670.00,690.00,710.00,845.0,0.0000,1.3067,1.8419
fico_range_high,1321847.0,699.0869,30.7998,614.0,674.00,694.00,714.00,850.0,0.0000,1.3069,1.8433
delinq_2yrs,1321818.0,0.3296,0.8920,0.0,0.00,0.00,0.00,39.0,0.0022,5.4097,53.9877
inq_last_6mths,1321817.0,0.6509,0.9577,0.0,0.00,0.00,1.00,33.0,0.0023,2.0674,10.4556
open_acc,1321818.0,11.6559,5.4694,0.0,8.00,11.00,14.00,97.0,0.0022,1.2995,3.3742
total_acc,1321818.0,25.0222,11.9035,1.0,16.00,23.00,32.00,176.0,0.0022,0.9460,1.6317


## 4. Histograms

Heavy-tailed columns (`annual_inc`, `revol_bal`, `tot_cur_bal`) are plotted on log1p.

In [7]:
plot_histograms(
    eda_reg, EDA_NUMERIC_REG,
    out_path=OUT_FIG / 'histograms_regression.png',
    bins=50,
    log_scale_cols=['annual_inc', 'revol_bal', 'tot_cur_bal', 'tot_hi_cred_lim'],
)

INFO | Saved histogram grid -> /Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados/outputs/figures/histograms_regression.png


In [8]:
plot_histograms(
    eda_clf, EDA_NUMERIC,
    out_path=OUT_FIG / 'histograms_classification.png',
    bins=50,
    log_scale_cols=['annual_inc', 'revol_bal', 'tot_cur_bal', 'tot_hi_cred_lim'],
)

INFO | Saved histogram grid -> /Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados/outputs/figures/histograms_classification.png


## 5. Class balance for the 3-class classification target

0 = Good, 1 = At risk, 2 = Default.

In [9]:
if 'target_class' in eda_clf.columns:
    counts = eda_clf['target_class'].value_counts().sort_index()
    pct = eda_clf['target_class'].value_counts(normalize=True).sort_index() * 100
    balance = pd.DataFrame({'count': counts, 'pct': pct.round(3)})
    balance.to_csv(OUT_TABLES / 'eda_class_balance.csv')
    print(balance)

    fig, ax = plt.subplots(figsize=(6, 4))
    balance['count'].plot(kind='bar', ax=ax, color=['#4C72B0','#DD8452','#C44E52'])
    ax.set_xticklabels(['0 Good', '1 At risk', '2 Default'], rotation=0)
    ax.set_title('Class balance (train+val)')
    fig.tight_layout()
    fig.savefig(OUT_FIG / 'class_balance.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

               count     pct
target_class                
0             899155  79.584
1              10102   0.894
2             220556  19.521


## 6. Relationship plots

- FICO vs `int_rate` (regression target).
- DTI vs `int_rate`.
- FICO vs `int_rate` coloured by `target_class` (to see how the two tasks interact).

In [10]:
if 'fico_range_low' in eda_reg.columns and 'int_rate' in eda_reg.columns:
    plot_scatter(eda_reg, x='fico_range_low', y='int_rate',
                 out_path=OUT_FIG / 'scatter_fico_vs_int_rate.png',
                 sample_n=20000)

INFO | Saved scatter -> /Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados/outputs/figures/scatter_fico_vs_int_rate.png


In [11]:
if 'dti' in eda_reg.columns and 'int_rate' in eda_reg.columns:
    plot_scatter(eda_reg, x='dti', y='int_rate',
                 out_path=OUT_FIG / 'scatter_dti_vs_int_rate.png',
                 sample_n=20000)

INFO | Saved scatter -> /Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados/outputs/figures/scatter_dti_vs_int_rate.png


## 7. Correlation heatmap

Spearman correlation (robust to non-linearity and outliers).

In [12]:
corr = plot_correlation_heatmap(
    eda_reg, EDA_NUMERIC_REG,
    out_path=OUT_FIG / 'corr_heatmap_regression.png',
    method='spearman',
)
corr.to_csv(OUT_TABLES / 'corr_spearman_regression.csv')
corr.round(2).head(10)

INFO | Saved heatmap -> /Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados/outputs/figures/corr_heatmap_regression.png


,loan_amnt,term,annual_inc,dti,fico_range_low,fico_range_high,delinq_2yrs,inq_last_6mths,open_acc,total_acc,...,revol_bal,revol_util,tot_cur_bal,tot_hi_cred_lim,bc_util,bc_open_to_buy,mo_sin_old_rev_tl_op,num_actv_rev_tl,pct_tl_nvr_dlq,int_rate
loan_amnt,1.00,0.43,0.50,0.05,0.13,0.13,0.01,-0.03,0.21,0.24,...,0.49,0.12,0.34,0.37,0.05,0.23,0.20,0.17,0.07,0.12
term,0.43,1.00,0.14,0.09,0.02,0.02,-0.00,-0.00,0.09,0.11,...,0.17,0.08,0.16,0.16,0.07,0.02,0.06,0.07,0.03,0.40
annual_inc,0.50,0.14,1.00,-0.21,0.11,0.11,0.09,0.03,0.25,0.32,...,0.41,0.07,0.53,0.56,0.01,0.22,0.25,0.13,-0.06,-0.13
dti,0.05,0.09,-0.21,1.00,-0.03,-0.03,-0.01,-0.00,0.32,0.24,...,0.25,0.17,0.12,0.12,0.17,-0.06,0.04,0.26,0.08,0.18
fico_range_low,0.13,0.02,0.11,-0.03,1.00,1.00,-0.21,-0.10,0.04,0.03,...,0.08,-0.36,0.17,0.24,-0.38,0.46,0.12,-0.15,0.35,-0.41
fico_range_high,0.13,0.02,0.11,-0.03,1.00,1.00,-0.21,-0.10,0.04,0.03,...,0.08,-0.36,0.17,0.24,-0.38,0.46,0.12,-0.15,0.35,-0.41
delinq_2yrs,0.01,-0.00,0.09,-0.01,-0.21,-0.21,1.00,0.02,0.06,0.13,...,-0.06,-0.02,0.09,0.09,-0.02,-0.05,0.11,-0.00,-0.49,0.05
inq_last_6mths,-0.03,-0.00,0.03,-0.00,-0.10,-0.10,0.02,1.00,0.13,0.14,...,-0.05,-0.08,0.03,0.03,-0.07,0.02,-0.01,0.11,-0.04,0.22
open_acc,0.21,0.09,0.25,0.32,0.04,0.04,0.06,0.13,1.00,0.70,...,0.38,-0.15,0.35,0.39,-0.12,0.30,0.15,0.65,0.05,-0.01
total_acc,0.24,0.11,0.32,0.24,0.03,0.03,0.13,0.14,0.70,1.00,...,0.31,-0.11,0.41,0.44,-0.09,0.23,0.34,0.38,-0.08,-0.05


## 8. Linear dimensionality reduction: PCA

Fit on a reproducible sample of the regression EDA set. Output: scree plot, 2D
projection, loadings table.

In [13]:
pca_info = run_pca(
    eda_reg, EDA_NUMERIC_REG,
    out_dir=OUT_FIG / 'pca',
    n_components=10,
    sample_n=50000,
    seed=42,
)
pd.Series(pca_info['explained_variance_ratio'],
          index=[f'PC{i+1}' for i in range(len(pca_info['explained_variance_ratio']))],
          name='explained_variance_ratio').to_csv(OUT_TABLES / 'pca_explained_variance.csv')
pd.Series(pca_info['cumulative'],
          index=[f'PC{i+1}' for i in range(len(pca_info['cumulative']))],
          name='cumulative').round(4)

PC1     0.1879
PC2     0.3362
PC3     0.4289
PC4     0.5119
PC5     0.5789
PC6     0.6407
PC7     0.6891
PC8     0.7331
PC9     0.7714
PC10    0.8052
Name: cumulative, dtype: float64

## 9. Non-linear dimensionality reduction: UMAP

Fit on the classification EDA set, coloured by `target_class`. If `umap-learn`
is not installed, install it with `pip install umap-learn`.

In [14]:
try:
    emb = run_umap(
        eda_clf, EDA_NUMERIC,
        out_dir=OUT_FIG / 'umap',
        hue='target_class',
        n_neighbors=30,
        min_dist=0.1,
        sample_n=30000,
        seed=42,
    )
    emb.to_parquet(PROJECT_ROOT / 'data' / 'processed' / 'umap_classification_sample.parquet')
    emb.head()
except ImportError as e:
    print('UMAP not available:', e)
    print('Install with: pip install umap-learn')

/Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


## 10. Outlier inspection

Quick IQR-based outlier counts per numeric column (documentation only — we do
not clip here; clipping is a learned transform and must be fit on training folds).

In [15]:
def iqr_outliers(s: pd.Series) -> int:
    s = s.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < lo) | (s > hi)).sum())

outliers = pd.Series({c: iqr_outliers(eda_reg[c]) for c in EDA_NUMERIC_REG})
outliers = outliers.sort_values(ascending=False)
outliers.to_csv(OUT_TABLES / 'eda_iqr_outliers_regression.csv')
outliers.head(15)

delinq_2yrs             262841
pub_rec                 218702
pub_rec_bankruptcies    153809
bc_open_to_buy          110641
revol_bal                77152
pct_tl_nvr_dlq           75848
inq_last_6mths           70905
annual_inc               65035
open_acc                 45684
tot_cur_bal              43163
tot_hi_cred_lim          42622
fico_range_low           39277
fico_range_high          39277
num_actv_rev_tl          37822
mo_sin_old_rev_tl_op     34974
dtype: int64

## 11. Initial insights (to be reused in the mid-journey report)

Write down, based on the plots above, at least:

1. Which numeric variables are heavy-tailed and require log or robust scaling.
2. Which variables correlate most with `int_rate` (excluding banned rate-determining fields).
3. Whether PCA suggests a small number of dominant dimensions.
4. Whether UMAP separates the three classes at all — a strong visual separation
   is informative for the expected classification difficulty.
5. Any outliers that need explicit handling in Phase 3.
6. Whether any apparent association could be confounded by cohort composition
   (e.g. riskier products issued in certain years).

Reminder: observational associations are not evidence of causation.